In [ ]:
import TechAna_DRAFT as TechAna
import pandas as pd
import requests
import numpy as np
import importlib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
INDUSTRY_MAP = {
    1: {'name': 'Banks','module': 'TotalScore_Bank'},
    2: {'name': 'Consumer','module': 'TotalScore_Consumer'},
    3: {'name': 'Financials','module': 'TotalScore_Financials'},
    4: {'name': 'Construction_and_materials','module': 'TotalScore_CM'},
    5: {'name': 'Goods_and_services','module': 'TotalScore_GS'},
    6: {'name': 'HealthCare','module': 'TotalScore_HealthCare'},
    7: {'name': 'Insurance','module': 'TotalScore_Insurance'},
    8: {'name': 'Materials','module': 'TotalScore_Materials'},
    9: {'name': 'RealEstate','module': 'TotalScore_RealEstate'},
    10: {'name': 'Utilities_and_Energy','module': 'TotalScore_UtiEne'},
    11: {'name': 'TechTele','module': 'TotalScore_TechTele'},
}

In [ ]:
def _parse_stock_payload(payload):
    records = []
    if isinstance(payload, list):
        records = payload
    elif isinstance(payload, dict):
        for sym, rows in payload.items():
            if isinstance(rows, list):
                for r in rows:
                    if 'symbol' not in r:
                        r = {**r, 'symbol': sym}
                    records.append(r)
            elif isinstance(rows, dict):
                if 'symbol' not in rows:
                    rows = {**rows, 'symbol': sym}
                records.append(rows)

    df = pd.DataFrame(records)
    if df.empty:
        return df

    for col in ['date', 'Date', 'trading_date', 'TradingDate']:
        if col in df.columns:
            df['date'] = pd.to_datetime(df[col], errors='coerce')
            break

    # Use Adj Close for all OHLC if available
    if 'adj_close' in df.columns:
        adj = pd.to_numeric(df['adj_close'], errors='coerce')
        df['open'] = adj
        df['high'] = adj
        df['low'] = adj
        df['close'] = adj

    else:
        for col in ['open', 'Open']:
            if col in df.columns:
                df['open'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['high', 'High']:
            if col in df.columns:
                df['high'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['low', 'Low']:
            if col in df.columns:
                df['low'] = pd.to_numeric(df[col], errors='coerce')
                break
        for col in ['close', 'Close']:
            if col in df.columns:
                df['close'] = pd.to_numeric(df[col], errors='coerce')
                break

    return df.dropna(subset=['symbol', 'date'])


def _compute_max_drawdown(series):
    if series is None or len(series) == 0:
        return 0.0
    running_max = series.cummax()
    drawdown = (series - running_max) / running_max
    return float(drawdown.min()) if len(drawdown) > 0 else 0.0


def _fetch_prices(symbols, start_date, end_date):
    if not symbols:
        return pd.DataFrame()
    params = {
        "symbols": ",".join(symbols),
        "start_date": start_date,
        "end_date": end_date
    }
    resp = requests.get(
        "http://192.168.8.190:8000/MKD/stock_daily",
        params=params,
        headers={"accept": "application/json"},
        timeout=30
    )
    resp.raise_for_status()
    payload = resp.json()
    return _parse_stock_payload(payload)


def _run_quarter_trades(symbols, start_date, end_date, df_prices, entry_override=None, original_entry_override=None):
    entry_override = entry_override or {}
    original_entry_override = original_entry_override or {}

    trades = []
    for sym in symbols:
        df_sym = df_prices[df_prices['symbol'] == sym].sort_values('date').reset_index(drop=True)
        if df_sym.empty:
            continue

        entry_row = df_sym.iloc[0]
        entry_date = entry_row['date']
        entry_price = entry_override.get(sym, entry_row.get('open', np.nan))
        if pd.isna(entry_price):
            continue

        original_entry_price = original_entry_override.get(sym, entry_price)

        sl_price = entry_price * 0.85
        tp_price = entry_price * 1.25

        exit_date = df_sym.iloc[-1]['date']
        exit_price = df_sym.iloc[-1]['close'] if 'close' in df_sym.columns else entry_price
        exit_reason = 'Keep Position'

        start_idx = 3 if len(df_sym) > 3 else len(df_sym)
        for i in range(start_idx, len(df_sym)):
            row = df_sym.iloc[i]
            day_open = row.get('open', np.nan)
            day_low = row.get('low', np.nan)
            day_high = row.get('high', np.nan)

            # Stoploss check (-15%)
            if pd.notna(day_low) and day_low <= sl_price:
                if pd.notna(day_open) and day_open <= sl_price:
                    exit_price = day_open
                else:
                    exit_price = sl_price
                exit_date = row['date']
                exit_reason = 'Stop Loss'
                break

            # Take profit check (+25%)
            if pd.notna(day_high) and day_high >= tp_price:
                exit_price = tp_price
                exit_date = row['date']
                exit_reason = 'Take Profit'
                break

        ret_pct = (exit_price - entry_price) / entry_price if entry_price else 0.0
        cum_ret_pct = (exit_price - original_entry_price) / original_entry_price if original_entry_price else 0.0

        max_dd = 0.0
        max_ret = 0.0
        min_ret = 0.0
        if 'close' in df_sym.columns:
            hold_df = df_sym[(df_sym['date'] >= entry_date) & (df_sym['date'] <= exit_date)]
            close_series = hold_df['close'].dropna()
            price_path = pd.concat([pd.Series([entry_price]), close_series], ignore_index=True)
            max_dd = _compute_max_drawdown(price_path)

            if not close_series.empty and entry_price:
                max_ret = (close_series.max() - entry_price) / entry_price
                min_ret = (close_series.min() - entry_price) / entry_price

        trades.append({
            'Symbol': sym,
            'Entry_Date': entry_date,
            'Entry_Price': entry_price,
            'Original_Entry_Price': original_entry_price,
            'Exit_Date': exit_date,
            'Exit_Price': exit_price,
            'Exit_Reason': exit_reason,
            'Return_Pct': ret_pct,
            'Max_Return_Pct': max_ret,
            'Min_Return_Pct': min_ret,
            'Cum_Return_Pct': cum_ret_pct,
            'Max_Drawdown': max_dd
        })

    return pd.DataFrame(trades)

def print_summary(df_trades, label, start_date, end_date, industry_name):
    if df_trades.empty:
        print(f'No trades generated for {label}.')
        return

    win_rate = (df_trades['Return_Pct'] > 0).mean()
    avg_returns = df_trades['Return_Pct'].mean()
    max_drawdown = df_trades['Max_Drawdown'].min()
    max_return = df_trades['Return_Pct'].max()
    min_return = df_trades['Return_Pct'].min()

    summary_df = pd.DataFrame([{
        'Industry': industry_name,
        'Label': label,
        'Period_Start': start_date,
        'Period_End': end_date,
        'Win_Rate': win_rate,
        'Average_Return': avg_returns,
        'Max_Return': max_return,
        'Min_Return': min_return,
        'Max_Drawdown': max_drawdown,
        'Deals': len(df_trades)
    }])

    print(f'\nTrade Results ({label}):')
    print(df_trades.to_string(index=False))
    print('\nSummary:')
    print(summary_df.to_string(index=False))

def run_backtest(industry_id):
    if industry_id not in INDUSTRY_MAP:
        print(f"Error: Industry ID {industry_id} not found in configuration.")
        return

    config = INDUSTRY_MAP[industry_id]
    industry_name = config['name']
    module_name = config['module']
    
    print(f"STARTING BACKTEST FOR: {industry_id} - {industry_name} (Module: {module_name})")
    try:
        TotalScore_Module = importlib.import_module(module_name)
    except ImportError:
        print(f"Error: Could not import module '{module_name}'. Check if file exists.")
        return

    TECH_START_DATE = TechAna.START_DATE
    TECH_END_DATE = TechAna.END_DATE

    tech_start = pd.to_datetime(TECH_START_DATE)
    tech_end = pd.to_datetime(TECH_END_DATE)
    tech_q = tech_end.to_period('Q')

    curr_q = tech_q + 1
    next_q = curr_q + 1

    PREV_START_DATE = tech_q.start_time.strftime('%Y-%m-%d')
    PREV_END_DATE = tech_q.end_time.strftime('%Y-%m-%d')
    START_DATE = curr_q.start_time.strftime('%Y-%m-%d')
    END_DATE = curr_q.end_time.strftime('%Y-%m-%d')
    NEXT_START_DATE = next_q.start_time.strftime('%Y-%m-%d')
    NEXT_END_DATE = next_q.end_time.strftime('%Y-%m-%d')

# Ranking for quarter t-1 (entry list for backtest quarter t)
    df_total_prev = TotalScore_Module.get_total_score(PREV_START_DATE, PREV_END_DATE, industry=industry_name)
    df_rank_prev = df_total_prev[['Symbol', 'Final_Score']].copy()
    df_rank_prev = df_rank_prev.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    TOP10_PREV = df_rank_prev.head(10)['Symbol'].tolist()

# Ranking for quarter t (used for rollover filter and next-quarter new entries)
    df_total_curr = TotalScore_Module.get_total_score(START_DATE, END_DATE, industry=industry_name)
    df_rank_curr = df_total_curr[['Symbol', 'Final_Score']].copy()
    df_rank_curr = df_rank_curr.sort_values('Final_Score', ascending=False).reset_index(drop=True)
    TOP10_CURR = df_rank_curr.head(10)['Symbol'].tolist()
    TOP13_CURR = df_rank_curr.head(13)['Symbol'].tolist()

    df_prices_curr = _fetch_prices(TOP10_PREV, START_DATE, END_DATE)
    df_trades_curr = _run_quarter_trades(TOP10_PREV, START_DATE, END_DATE, df_prices_curr)

    rollover_symbols = []
    entry_override_next = {}
    original_entry_override_next = {}

    if not df_trades_curr.empty:
        pending_df = df_trades_curr[df_trades_curr['Exit_Reason'] == 'Keep Position']
        for _, row in pending_df.iterrows():
            sym = row['Symbol']
            if sym in TOP13_CURR:
                rollover_symbols.append(sym)
                original_entry_price = row.get('Original_Entry_Price', row['Entry_Price'])
                if row.get('Return_Pct', 0) > 0:
                    reference_price = row['Exit_Price']
                else:
                    reference_price = original_entry_price

                entry_override_next[sym] = reference_price
                original_entry_override_next[sym] = original_entry_price
                df_trades_curr.loc[df_trades_curr['Symbol'] == sym, 'Exit_Reason'] = 'Rollover'

    print_summary(df_trades_curr, f"{curr_q.year}Q{curr_q.quarter}", START_DATE, END_DATE, industry_name)

    symbols_next = TOP10_CURR + [s for s in rollover_symbols if s not in TOP10_CURR]
    df_prices_next = _fetch_prices(symbols_next, NEXT_START_DATE, NEXT_END_DATE)
    df_trades_next = _run_quarter_trades(
        symbols_next,
        NEXT_START_DATE,
        NEXT_END_DATE,
        df_prices_next,
        entry_override=entry_override_next,
        original_entry_override=original_entry_override_next
    )

    print_summary(df_trades_next, f"{next_q.year}Q{next_q.quarter}", NEXT_START_DATE, NEXT_END_DATE, industry_name)

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 1 - Banks (Module: TotalScore_Bank)
[Banks] Mega Caps: ['VCB', 'BID', 'CTG', 'TCB']
[Banks] Large Caps: ['VPB', 'MBB', 'HDB', 'ACB', 'LPB', 'STB']
[Banks] Mega Caps: ['VCB', 'BID', 'CTG', 'TCB']
[Banks] Large Caps: ['VPB', 'MBB', 'HDB', 'ACB', 'LPB', 'STB']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   TCB 2024-07-01     22065.63              22065.63 2024-09-30    23575.64      Rollover    0.068433        0.068433       -0.079470        0.068433     -0.120253
   ACB 2024-07-01     19887.28              19887.28 2024-09-30    21516.70      Rollover    0.081933        0.100840       -0.025210        0.081933     -0.075697
   STB 2024-07-01     29300.00              29300.00 2024-09-30    33350.00      Rollover    0.138225        0.146758       -0.071672        0.138225     -0.111111
   MBB 2024-07-01     14628.42        

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 2 - Consumer (Module: TotalScore_Consumer)
Success
[Consumer] Mega Caps: ['MCH', 'VNM', 'MWG', 'MSN']
[Consumer] Large Caps: ['VJC', 'HVN', 'SAB', 'PNJ', 'FRT', 'HAG']
[Consumer] Mega Caps: ['MCH', 'VNM', 'MWG', 'MSN']
[Consumer] Large Caps: ['VJC', 'HVN', 'SAB', 'PNJ', 'FRT', 'HAG']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   ADS 2024-07-01     11147.36              11147.36 2024-07-23     9382.72     Stop Loss   -0.158301        0.027027       -0.158301       -0.158301     -0.180451
   MSH 2024-07-01     27108.84              27108.84 2024-09-30    26324.70      Rollover   -0.028926        0.049587       -0.109504       -0.028926     -0.138000
   HAG 2024-07-01     12200.00              12200.00 2024-08-14    10050.00     Stop Loss   -0.176230        0.016393       -0.176230       -0.176230     -0.189516
   SAB 2024

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 3 - Financials (Module: TotalScore_Financials)
Success
[Financials] Mega Caps: ['SSI', 'VIX', 'VND', 'VCI']
[Financials] Large Caps: ['HCM', 'MBS', 'SHS', 'FTS', 'BSI', 'EVF']
[Financials] Mega Caps: ['SSI', 'VIX', 'VND', 'VCI']
[Financials] Large Caps: ['HCM', 'MBS', 'SHS', 'FTS', 'BSI', 'EVF']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   MBS 2024-07-01     23274.42              23274.42 2024-09-30    27963.54      Rollover    0.201471        0.201471       -0.087638        0.201471     -0.180421
   TVS 2024-07-01     19108.33              19108.33 2024-08-01    16032.96     Stop Loss   -0.160944        0.072961       -0.160944       -0.160944     -0.218000
   CTS 2024-07-01     27668.52              27668.52 2024-08-01    22672.82     Stop Loss   -0.180555        0.064394       -0.180555       -0.180555     -0.230130

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 4 - Construction_and_materials (Module: TotalScore_CM)
Success
[Construction_and_materials] Mega Caps: ['VGC', 'CC1', 'BMP', 'LGC']
[Construction_and_materials] Large Caps: ['VCG', 'CII', 'CTR', 'NTP', 'PC1', 'CTD']
[Construction_and_materials] Mega Caps: ['VGC', 'CC1', 'BMP', 'LGC']
[Construction_and_materials] Large Caps: ['VCG', 'CII', 'CTR', 'NTP', 'PC1', 'CTD']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   VCS 2024-07-01     65566.85              65566.85 2024-08-05    55637.90     Stop Loss   -0.151432        0.070941       -0.151432       -0.151432     -0.207643
   DPG 2024-07-01     34294.49              34294.49 2024-08-05    28075.92     Stop Loss   -0.181329        0.078995       -0.181329       -0.181329     -0.241265
   NTP 2024-07-01     41506.64              41506.64 2024-08-16    51883.30   Take Profit  

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 5 - Goods_and_services (Module: TotalScore_GS)
Success
[Goods_and_services] Mega Caps: ['ACV', 'MVN', 'GEE', 'VEA']
[Goods_and_services] Large Caps: ['GEX', 'GMD', 'VTP', 'PHP', 'PVT', 'HAH']
[Goods_and_services] Mega Caps: ['ACV', 'MVN', 'GEE', 'VEA']
[Goods_and_services] Large Caps: ['GEX', 'GMD', 'VTP', 'PHP', 'PVT', 'HAH']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   VIP 2024-07-01     12230.82              12230.82 2024-07-10   15288.525   Take Profit    0.250000        0.250950       -0.022814        0.250000     -0.022814
   PVT 2024-07-01     22130.90              22130.90 2024-09-30   21197.280      Rollover   -0.042186        0.040678       -0.101695       -0.042186     -0.136808
   GEX 2024-07-01     20476.68              20476.68 2024-09-30   20016.530 Keep Position   -0.022472        0.065168       -0.0876

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 6 - HealthCare (Module: TotalScore_HealthCare)
['BCP', 'BIO', 'CNC', 'DBM', 'DHN', 'DPH', 'DPP', 'DTH', 'HDP', 'MRF', 'MTP', 'NDC', 'NDP', 'NTF', 'YTC']
Symbols with data: 28 / 43
Success
[HealthCare] Mega Caps: ['DHG', 'IMP', 'DHT', 'DVN']
[HealthCare] Large Caps: ['DBD', 'DCL', 'DTP', 'TRA', 'DMC', 'TNH']
[HealthCare] Mega Caps: ['DHG', 'IMP', 'DHT', 'DVN']
[HealthCare] Large Caps: ['DBD', 'DCL', 'DTP', 'TRA', 'DMC', 'TNH']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DHG 2024-07-01     99817.30              99817.30 2024-09-30  97428.8700      Rollover   -0.023928        0.063548       -0.038510       -0.023928     -0.095960
   MKP 2024-07-01     29981.88              29981.88 2024-09-30  28904.1000 Keep Position   -0.035948        0.094771       -0.052288       -0.035948     -0.134328
   AGP 2024-07-01     34244.00

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 7 - Insurance (Module: TotalScore_Insurance)
Success
[Insurance] Mega Caps: ['BVH', 'PVI', 'BIC', 'VNR']
[Insurance] Large Caps: ['MIG', 'PTI', 'BMI', 'PRE', 'PGI', 'ABI']
[Insurance] Mega Caps: ['BVH', 'PVI', 'BIC', 'VNR']
[Insurance] Large Caps: ['MIG', 'PTI', 'BMI', 'PRE', 'PGI', 'ABI']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   PVI 2024-07-01     51686.75              51686.75 2024-09-10    43279.00   Stop Loss   -0.162667        0.005217       -0.162667       -0.162667     -0.167013
   PRE 2024-07-01     16094.68              16094.68 2024-09-30    15921.36    Rollover   -0.010769        0.054455       -0.016204       -0.010769     -0.067010
   MIG 2024-07-01     18383.58              18383.58 2024-08-05    14993.70   Stop Loss   -0.184397        0.115839       -0.184397       -0.184397     -0.269068
   BVH 2024-0

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 8 - Materials (Module: TotalScore_Materials)
Success
[Materials] Mega Caps: ['HPG', 'GVR', 'KSV', 'MSR']
[Materials] Large Caps: ['DGC', 'DCM', 'DPM', 'HSG', 'PHR', 'NKG']
[Materials] Mega Caps: ['HPG', 'GVR', 'KSV', 'MSR']
[Materials] Large Caps: ['DGC', 'DCM', 'DPM', 'HSG', 'PHR', 'NKG']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   DPM 2024-07-01     18997.76              18997.76 2024-09-30    19711.72      Rollover    0.037581        0.073204       -0.096685        0.037581     -0.158301
   HPG 2024-07-01     23607.05              23607.05 2024-09-30    21941.65      Rollover   -0.070547        0.022927       -0.123457       -0.070547     -0.143103
   GVR 2024-07-01     33507.67              33507.67 2024-09-30    34924.18      Rollover    0.042274        0.122449       -0.122449        0.042274     -0.218182
   VG

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 9 - RealEstate (Module: TotalScore_RealEstate)
Success
Starting Technical & Risk Analysis for 50 symbols...
[RealEstate] Mega Caps: ['VIC', 'VHM', 'BCM', 'VRE']
[RealEstate] Large Caps: ['KSF', 'KBC', 'KDH', 'NVL', 'VEF', 'PDR']
Starting Technical & Risk Analysis for 50 symbols...
[RealEstate] Mega Caps: ['VIC', 'VHM', 'BCM', 'VRE']
[RealEstate] Large Caps: ['KSF', 'KBC', 'KDH', 'NVL', 'VEF', 'PDR']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   BCM 2024-07-01     62153.50              62153.50 2024-09-30    69147.00 Keep Position    0.112520        0.161648        0.000000        0.112520     -0.079127
   LHG 2024-07-01     34371.00              34371.00 2024-09-30    34842.24 Keep Position    0.013710        0.156952       -0.002817        0.013710     -0.138095
   TCH 2024-07-01     15619.90              15619.90 2024

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 10 - Utilities_and_Energy (Module: TotalScore_UtiEne)
Success
[Utilities_and_Energy] Mega Caps: ['GAS', 'BSR', 'PLX', 'POW']
[Utilities_and_Energy] Large Caps: ['REE', 'PGV', 'PVS', 'PVD', 'OIL', 'VSH']
[Utilities_and_Energy] Mega Caps: ['GAS', 'BSR', 'PLX', 'POW']
[Utilities_and_Energy] Large Caps: ['REE', 'PGV', 'PVS', 'PVD', 'OIL', 'VSH']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price   Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   BSR 2024-07-01     13290.94              13290.94 2024-09-30    14493.74      Rollover    0.090498        0.122172       -0.036199        0.090498     -0.119835
   PVS 2024-07-01     37621.60              37621.60 2024-09-30    37254.56      Rollover   -0.009756        0.078049       -0.078049       -0.009756     -0.144796
   PVB 2024-07-01     27600.00              27600.00 2024-09-30    29500.00 Keep Position    0.068841        0.14492

In [ ]:
if __name__ == "__main__":
    try:
        selection = int(input("Enter Industry ID to Backtest: "))
        run_backtest(selection)
    except ValueError:
        print("Invalid input. Please enter a number.")

STARTING BACKTEST FOR: 11 - TechTele (Module: TotalScore_TechTele)
Success
[TechTele] Mega Caps: ['VGI', 'FPT', 'FOX', 'CMG']
[TechTele] Large Caps: ['ELC', 'SGT', 'VEC', 'TTN', 'ICT', 'POT']
[TechTele] Mega Caps: ['VGI', 'FPT', 'FOX', 'CMG']
[TechTele] Large Caps: ['ELC', 'SGT', 'VEC', 'TTN', 'ICT', 'POT']

Trade Results (2024Q3):
Symbol Entry_Date  Entry_Price  Original_Entry_Price  Exit_Date  Exit_Price Exit_Reason  Return_Pct  Max_Return_Pct  Min_Return_Pct  Cum_Return_Pct  Max_Drawdown
   ELC 2024-07-01     20595.08              20595.08 2024-09-30    20720.40    Rollover    0.006085        0.054766       -0.113590        0.006085     -0.159615
   CMG 2024-07-01     51005.01              51005.01 2024-08-01    41917.44   Stop Loss   -0.178170        0.062600       -0.178170       -0.178170     -0.226586
   FPT 2024-07-01    109014.22             109014.22 2024-09-30   114015.65    Rollover    0.045879        0.085537       -0.077760        0.045879     -0.150430
   ICT 2024-07-01 